# 02 — VAR dynamics and rolling residuals

Fit the interpretable conditional-mean model and construct pseudo-out-of-sample calibration errors. The test period remains untouched.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from innovcal.data import chronological_split
from innovcal.di_var import rolling_var_residuals
from innovcal.innovations.diagnostics import summarize_innovations
from innovcal.vector_ar.fit import fit_var_ols
from innovcal.vector_ar.stability import stability_summary

In [2]:
returns = pd.read_csv(ROOT / 'data/processed/financial_returns.csv', index_col=0, parse_dates=True)
split = chronological_split(returns.to_numpy(), 0.6, 0.2)
LAGS = 1
calibration_sample = np.vstack([split.train, split.calibration])
residuals = rolling_var_residuals(calibration_sample, len(split.train), lags=LAGS)
var_fit = fit_var_ols(calibration_sample, lags=LAGS, include_intercept=True)
print('train/calibration/test:', len(split.train), len(split.calibration), len(split.test))
print('rolling residuals:', residuals.shape)

train/calibration/test: 359 119 121
rolling residuals: (119, 4)


/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta


In [3]:
k = returns.shape[1]
beta_no_intercept = var_fit['beta'][1:]
print(stability_summary(beta_no_intercept, k=k, lags=LAGS))
diagnostics = summarize_innovations(residuals)
correlation = diagnostics.pop('correlation')
display(pd.DataFrame(diagnostics, index=returns.columns))
display(pd.DataFrame(correlation, index=returns.columns, columns=returns.columns))

{'stable': True, 'eigenvalues': array([-0.15323247+0.j        ,  0.06576739+0.j        ,
        0.01375369+0.06748786j,  0.01375369-0.06748786j]), 'max_modulus': 0.1532324668240954}


,mean,std,skewness,kurtosis,excess_kurtosis,jarque_bera_stat,jarque_bera_pvalue
Asset_1,-0.000669,0.007840,-0.167096,2.455603,-0.544397,2.160943,0.339435
Asset_2,0.000451,0.010437,-0.033808,3.422515,0.422515,0.646868,0.723660
Asset_3,-0.000928,0.010769,-0.033113,3.097610,0.097610,0.030607,0.984813
Asset_4,-0.000298,0.016113,0.017695,2.956491,-0.043509,0.047750,0.976408


,Asset_1,Asset_2,Asset_3,Asset_4
Asset_1,1.000000,0.362878,0.308206,0.208500
Asset_2,0.362878,1.000000,0.305455,0.338840
Asset_3,0.308206,0.305455,1.000000,0.266065
Asset_4,0.208500,0.338840,0.266065,1.000000


In [4]:
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
np.savez(
    CACHE / 'var_stage.npz',
    train=split.train, calibration=split.calibration, test=split.test,
    residuals=residuals, beta=var_fit['beta'], lags=LAGS,
)